In [ ]:
from langgraph.graph import StateGraph, START, END
from pydantic import BaseModel, Field
from langgraph.checkpoint.memory import InMemorySaver


# =========================================================
# 1. WORKER MODEL
# =========================================================

class Worker(BaseModel):
    name: str
    status: str = "PENDING"
    dependency: list[str] = Field(default_factory=list)


# =========================================================
# 2. GLOBAL STATE
# =========================================================

class NodeState(BaseModel):

    name: str = ""
    input: str = ""

    workers: list[Worker] = Field(
        default_factory=list
    )

    completed_task: list[str] = Field(
        default_factory=list
    )


# =========================================================
# 3. CREATE TASKS
# =========================================================

def create_task(state: NodeState):

    workers = [

        Worker(
            name="First_Node",
            status="PENDING",
            dependency=[]
        ),

        Worker(
            name="Second_Node",
            status="PENDING",
            dependency=["First_Node"]
        )
    ]

    print("\n========== TASKS CREATED ==========\n")

    for task in workers:
        print(
            f"{task.name:15} | "
            f"Dependency: {task.dependency} | "
            f"Status: {task.status}"
        )

    return {
        "workers": workers
    }


# =========================================================
# 4. SCHEDULER
# =========================================================

def scheduler(state: NodeState):

    print("\n========== SCHEDULER ==========\n")

    ready_task = []

    concurrency_limit = 2

    # -----------------------------------------------------
    # Find READY tasks
    # -----------------------------------------------------

    for task in state.workers:

        # Already completed
        if task.name in state.completed_task:
            continue

        # Check dependencies
        dependency_satisfied = all(
            dependency in state.completed_task
            for dependency in task.dependency
        )

        if dependency_satisfied:

            task.status = "READY"
            ready_task.append(task.name)

        else:

            task.status = "BLOCKED"

    # -----------------------------------------------------
    # Apply concurrency limit
    # -----------------------------------------------------

    running_task = ready_task[:concurrency_limit]

    waiting_task = ready_task[concurrency_limit:]

    # -----------------------------------------------------
    # RUNNING
    # -----------------------------------------------------

    for task in state.workers:

        if task.name in running_task:

            task.status = "RUNNING"

        elif task.name in waiting_task:

            task.status = "WAITING"

    # -----------------------------------------------------
    # Print scheduler state
    # -----------------------------------------------------

    for task in state.workers:

        print(
            f"{task.name:15} | "
            f"Dependency: {task.dependency} | "
            f"Status: {task.status}"
        )

    return {
        "workers": state.workers
    }


# =========================================================
# 5. RUNTIME
# =========================================================

def runtime(state: NodeState):

    print("\n========== RUNTIME ==========\n")

    completed_now = []

    for task in state.workers:

        # Only RUNNING tasks execute
        if task.status != "RUNNING":
            continue

        print(
            f"Executing: {task.name}"
        )

        # Simulate successful execution
        task.status = "SUCCESS"

        completed_now.append(
            task.name
        )

        print(
            f"{task.name} -> SUCCESS"
        )

    return {

        "workers": state.workers,

        "completed_task": completed_now

    }


# =========================================================
# 6. REMAINING TASK ROUTER
# =========================================================

def remaining_task(state: NodeState):

    for task in state.workers:

        if task.name not in state.completed_task:

            return "Scheduler"

    return "END"


# =========================================================
# 7. CHECKPOINTER
# =========================================================

checkpointer = InMemorySaver()


# =========================================================
# 8. THREAD CONFIG
# =========================================================

config = {
    "configurable": {
        "thread_id": "thread_001"
    }
}


# =========================================================
# 9. GRAPH
# =========================================================

graph = StateGraph(NodeState)

graph.add_node(
    "Create_Task",
    create_task
)

graph.add_node(
    "Schedule",
    scheduler
)

graph.add_node(
    "Runtime",
    runtime
)


# =========================================================
# 10. GRAPH FLOW
# =========================================================

graph.add_edge(
    START,
    "Create_Task"
)

graph.add_edge(
    "Create_Task",
    "Schedule"
)

graph.add_edge(
    "Schedule",
    "Runtime"
)


# =========================================================
# 11. CONDITIONAL ROUTING
# =========================================================

graph.add_conditional_edges(
    "Runtime",
    remaining_task,
    {
        "Scheduler": "Schedule",
        "END": END
    }
)


# =========================================================
# 12. COMPILE WITH CHECKPOINTER
# =========================================================

app = graph.compile(
    checkpointer=checkpointer
)


# =========================================================
# 13. FIRST EXECUTION
# =========================================================

result = app.invoke(
    {},
    config=config
)


print("\n========== FINAL RESULT ==========\n")

print(result)


# =========================================================
# 14. GET SAVED CHECKPOINT STATE
# =========================================================

saved_state = app.get_state(
    config
)


print("\n========== SAVED STATE ==========\n")

print(saved_state)


# =========================================================
# 15. CHECK THREAD ID
# =========================================================

print("\n========== THREAD ID ==========\n")

print(
    saved_state.config["configurable"]["thread_id"]
)


# =========================================================
# 16. CHECK CURRENT VALUES
# =========================================================

print("\n========== CURRENT VALUES ==========\n")

print(
    "Completed Tasks:",
    saved_state.values["completed_task"]
)

for task in saved_state.values["workers"]:

    print(
        f"{task.name} -> {task.status}"
    )


========== TASKS CREATED ==========

First_Node      | Dependency: [] | Status: PENDING
Second_Node     | Dependency: ['First_Node'] | Status: PENDING

========== SCHEDULER ==========

First_Node      | Dependency: [] | Status: RUNNING
Second_Node     | Dependency: ['First_Node'] | Status: BLOCKED

========== RUNTIME ==========

Executing: First_Node
First_Node -> SUCCESS

========== SCHEDULER ==========

First_Node      | Dependency: [] | Status: SUCCESS
Second_Node     | Dependency: ['First_Node'] | Status: RUNNING

========== RUNTIME ==========

Executing: Second_Node
Second_Node -> SUCCESS

========== SCHEDULER ==========

First_Node      | Dependency: [] | Status: RUNNING
Second_Node     | Dependency: ['First_Node'] | Status: SUCCESS

========== RUNTIME ==========

Executing: First_Node
First_Node -> SUCCESS

========== SCHEDULER ==========

First_Node      | Dependency: [] | Status: SUCCESS
Second_Node     | Dependency: ['First_Node'] | Status: RUNNING

========== RUNTIME ======

GraphRecursionError: Recursion limit of 10007 reached without hitting a stop condition. You can increase the limit by setting the `recursion_limit` config key.
For troubleshooting, visit: https://docs.langchain.com/oss/python/langgraph/errors/GRAPH_RECURSION_LIMIT